In [25]:
#imports
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import random
import json 


In [26]:
driver = webdriver.Chrome()
driver.get("https://www.imdb.com/chart/top/?ref_=hm_nv_menu")

In [27]:
elems = driver.find_elements(By.CLASS_NAME, "ipc-metadata-list-summary-item")
# elems is a list of webelements, these are kind of pointers to elements of a html


In [28]:
print(len(elems))

250


In [ ]:

movie_links=[]
# 125 movies
for elem in elems[0:125]:
    d = elem.get_attribute("outerHTML")
    soup = BeautifulSoup(d, 'html.parser')
    partial_movie_link = soup.find('a', class_= "ipc-lockup-overlay ipc-focusable ipc-focusable--constrained").get('href')
    movie_link = f"https://www.imdb.com{partial_movie_link}"
    movie_links.append(movie_link)
    
driver.close()        
    

In [30]:
driver = webdriver.Chrome()
for index in range(len(movie_links)):
    wait = WebDriverWait(driver, 10)
    driver.get(movie_links[index])

    # Scroll down
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # Now wait for Tech specs
    wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//span[text()='Tech specs']")
    )
    ) 

    outer_html = driver.page_source
    with open(f'movie_{index}.html', 'w', encoding='utf-8') as f:
        f.write(outer_html)
driver.close()    
  
    



In [31]:
def safe_text(elem):
    return elem.get_text(strip=True) if elem else None


In [61]:
#parsing each movie's html
def parse_movie(html):
   with open(html, encoding="utf-8") as f:
      soup = BeautifulSoup(f,'html.parser')
   data = {}
   # extracting data using css selectors
   # Title is h1 tag's text 
   title_tag = soup.find('h1')
   data['title'] = safe_text(title_tag)
   # duration can be easily accesssed using data-testid 
   duration_tag = soup.find("li", {"data-testid" : "title-techspec_runtime" })
   data["duration"] = safe_text(duration_tag).replace("Runtime","")
   # rating 
   rating_tag = soup.find("div",{"data-testid": "hero-rating-bar__aggregate-rating__score"})
   rating_text = safe_text(rating_tag)
   data["rating"] = rating_text.split("/")[0] if rating_text else None
   # genres are texts in class ipc-chip__text 
   genres_tags = soup.find_all("span",class_="ipc-chip__text")
   data["genres"] = []
   for i in range(5):
    data["genres"].append(safe_text(genres_tags[i]))
    
   credit_cards = soup.find_all("li",{"data-testid":"title-pc-principal-credit"})
   
   # directors     
   directors = credit_cards[0].find_all("a") if credit_cards else []
   data["directors"] = [safe_text(elem) for elem in directors]
   # writers
   writers = credit_cards[1].find_all("a") if credit_cards else []
   data["writers"] = [safe_text(elem) for elem in writers]
  
   # cast(top 6)
   cast = soup.find_all('a', {"data-testid":"title-cast-item__actor"})
   data["cast"] = [safe_text(elem) for elem in cast[0:6]]
   
   # summary 
   summary_block = soup.find('div', {"data-testid": "ai-review-summary-text"})

   if summary_block:
    inner = summary_block.find('div', class_="ipc-html-content-inner-div")
    data["summary"] = safe_text(inner)
   else:
    data["summary"] = None
    
   # Poster
   poster_tag = soup.find('div',{"data-testid":"hero-media__poster"})
   poster = poster_tag.find('a').get('href')
   data["poster"] = (f"https://www.imdb.com{poster}") 
   # videos
   videos_section = soup.find('section',{"data-testid":"videos-section"}) 
   trailer = videos_section.find('a',{"data-testid":"videos-slate-overlay-1"}) 
   data["trailer"] = f"https://www.imdb.com{trailer.get('href')}"
   # box office
   box_office_block =  soup.find('section',{"data-testid":"BoxOffice"})
   
   budget_tag = box_office_block.find('li',{"data-testid":"title-boxoffice-budget"}) 
   budget = budget_tag.find('span',class_="ipc-metadata-list-item__list-content-item ipc-btn--not-interactable")
   if budget:
    data["budget"] = safe_text(budget)
   else:
    data["budget"] = None  
   collection_tag = box_office_block.find('li',{"data-testid":"title-boxoffice-cumulativeworldwidegross"})
   collection = collection_tag.find('span',class_="ipc-metadata-list-item__list-content-item ipc-btn--not-interactable")
   if collection:
    data["worldwide_collection"] = safe_text(collection)
   else:
      data["worldwide_collection"] = None 
   return data


In [62]:
movies_data = []

for i in range(len(movie_links)):  
    movie = parse_movie(f"movie_{i}.html")
    movies_data.append(movie)

with open("movies.json", "w", encoding="utf-8") as f:
    json.dump(movies_data, f, indent=4, ensure_ascii=False)
